In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer: una Qiskit Function de Global Data Quantum


*Consulta la [referencia de la API](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Las Qiskit Functions son una funcionalidad experimental disponible únicamente para usuarios del Plan Premium, Plan Flex y Plan On-Prem (a través de la API de IBM Quantum Platform) de IBM Quantum&reg;. Se encuentran en estado de versión preliminar y pueden cambiar.
## Descripción general
El Quantum Portfolio Optimizer es una Qiskit Function que aborda el problema de optimización dinámica de carteras, un problema estándar en finanzas que busca rebalancear inversiones periódicas en un conjunto de activos para maximizar rendimientos y minimizar riesgos. Al desplegar técnicas de optimización cuántica de vanguardia, esta función simplifica el proceso para que los usuarios, sin experiencia en computación cuántica, puedan beneficiarse de sus ventajas a la hora de encontrar trayectorias de inversión óptimas. Ideal para gestores de carteras, investigadores en finanzas cuantitativas e inversores individuales, esta herramienta permite realizar backtesting de estrategias de negociación en optimización de carteras.
## Descripción de la función
La función Quantum Portfolio Optimizer utiliza el algoritmo Variational Quantum Eigensolver (VQE) para resolver un problema de Optimización Binaria Cuadrática sin Restricciones (QUBO), abordando problemas de optimización dinámica de carteras. Los usuarios solo necesitan proporcionar los datos de precios de los activos y definir la restricción de inversión; luego, la función ejecuta el proceso de optimización cuántica que devuelve un conjunto de trayectorias de inversión optimizadas.

El proceso consta de cuatro etapas principales. Primero, los datos de entrada se mapean a un problema compatible con la computación cuántica, construyendo el QUBO del problema de optimización dinámica de carteras y transformándolo en un operador cuántico (hamiltoniano de Ising). A continuación, el problema de entrada y el algoritmo VQE se adaptan para ejecutarse en hardware cuántico. El algoritmo VQE se ejecuta entonces en el hardware cuántico y, finalmente, los resultados se post-procesan para proporcionar las trayectorias de inversión óptimas. El sistema también incluye un post-procesamiento consciente del ruido (basado en [SQD](/guides/qiskit-addons-sqd)) para maximizar la calidad de la salida.

Esta Qiskit Function está basada en el [artículo publicado](https://arxiv.org/abs/2412.19150) por Global Data Quantum.
![Visualización del flujo de trabajo de la función](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Comenzar
Autentícate con tu [clave de API](https://quantum.cloud.ibm.com/) y selecciona la Qiskit Function de la siguiente manera. (Este fragmento asume que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## Ejemplo: Optimización dinámica de carteras con siete activos
Este ejemplo demuestra cómo ejecutar la función de optimización dinámica de carteras (DPO) y ajustar su configuración para un rendimiento óptimo. Incluye pasos detallados para ajustar los parámetros y lograr los resultados deseados.

Este caso implica siete activos, cuatro pasos temporales y cuatro qubits de resolución, lo que resulta en un requisito total de 112 qubits.
### 1. Lee los activos incluidos en la cartera
Si todos los activos de la cartera están almacenados en una carpeta en una ruta específica, puedes cargarlos en un `pandas.DataFrame` y convertirlos a un objeto en formato `dict` usando la siguiente función.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

Para este ejemplo, hemos utilizado los activos [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) y [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). La siguiente figura ilustra los datos utilizados en este ejemplo, mostrando la evolución diaria del precio de cierre de los activos del 1 de enero al 1 de septiembre de 2023.

En este ejemplo, para garantizar la uniformidad entre fechas, hemos rellenado los días no bursátiles con el precio de cierre de la fecha disponible anterior. Aplicamos este paso porque los activos seleccionados provienen de diferentes mercados con distintos días de negociación, lo que hace esencial estandarizar el conjunto de datos para mantener la consistencia.
![Visualización de los datos históricos de los activos](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Define el problema
Define las especificaciones del problema configurando los parámetros en el diccionario `qubo_settings`.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. Define la configuración del optimizador y del ansatz (Opcional)
Opcionalmente, define requisitos específicos para el proceso de optimización, incluida la selección del optimizador y sus parámetros, así como la especificación del primitivo y sus configuraciones.

Para el Tailored Ansatz, el tamaño de población elegido se basó en experimentos previos que muestran que este valor produce una optimización estable y eficiente.

En el caso del Real Amplitudes Ansatz, puedes seguir una relación lineal entre el ``population_size`` y el número de qubits en el circuito. Como regla general aproximada, se recomienda usar un **mínimo** de ``population_size ~ 0.8 * n_qubits`` para el ansatz `real_amplitudes`.

Se espera que el Optimized Real Amplitudes tenga un mejor rendimiento de optimización que el ansatz Real Amplitudes. Sin embargo, el número de variables a optimizar en este ansatz crece mucho más rápido que en el caso de Real Amplitudes (consulta el [artículo](https://arxiv.org/pdf/2412.19150)). Por lo tanto, para problemas grandes, el Optimized Real Amplitudes requiere más ejecuciones de circuitos. Es probable que el Optimized Real Amplitudes sea útil para problemas de hasta 100 qubits, pero se recomienda ser cuidadoso al establecer los parámetros ``population_size``. Como ejemplo de este escalado en ``population_size``, la tabla anterior muestra que para un problema de 84 qubits, el Optimized Real Amplitudes requiere 120 de ``population_size``, mientras que para un problema de 56 qubits, un ``population_size`` de 40 es suficiente.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

También es posible elegir un ansatz específico. El siguiente código usa el ansatz ``'Tailored'``.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. Ejecuta el problema

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Recupera los resultados
La función devuelve un diccionario con las trayectorias de inversión ordenadas de menor a mayor según su valor de función objetivo (consulta la sección [Salida](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) de la referencia de la API). Este conjunto de resultados permite identificar la trayectoria con el menor coste y sus evaluaciones de inversión correspondientes. Además, permite analizar diferentes trayectorias, facilitando la selección de las que mejor se adaptan a necesidades u objetivos específicos. Esta flexibilidad garantiza que las decisiones puedan adaptarse a una variedad de preferencias o escenarios.
Comienza presentando la estrategia resultante que logró el menor coste objetivo encontrado durante el proceso.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


A continuación, usando los metadatos, puedes acceder a los resultados de todas las estrategias muestreadas. Así puedes analizar con mayor detalle las trayectorias alternativas devueltas por el optimizador. Para ello, lee el diccionario almacenado en `dpo_result['metadata']['all_samples_metrics']`, que contiene no solo información adicional sobre la estrategia óptima, sino también detalles de las otras estrategias candidatas evaluadas durante la optimización.

El siguiente ejemplo muestra cómo leer esta información usando `pandas` para extraer métricas clave asociadas a la estrategia óptima. Estas incluyen la Desviación de Restricción, el Ratio de Sharpe y el rendimiento de inversión correspondiente.